<a href="https://colab.research.google.com/github/mariemtnb/crop-disease-PDL/blob/main/crop_disease_detection_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌿 Crop Disease Detection — Full Computer Vision Pipeline
**Dataset:** New Plant Diseases Dataset | **Dr. Ing. Hela Mahersia — SESAME**

This notebook applies **all techniques from Chapters 2–5** before training a CNN:
- Ch.2 → Image Enhancement & Restoration (histograms, linear/non-linear filtering)
- Ch.3 → Mathematical Morphology (erosion, dilation, opening, closing, tophat)
- Ch.4 → Image Segmentation (Otsu, Canny/Sobel/Laplacian, K-means)
- Ch.5 → Texture Analysis (GLCM/Haralick, LBP, wavelets)
- CNN from scratch + Transfer Learning (EfficientNetB3)

## 1. Setup & Imports

In [3]:
import os, warnings, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
warnings.filterwarnings('ignore')

import cv2
from PIL import Image
from scipy import ndimage
import pywt  # wavelets

from skimage import filters, morphology, feature, measure
from skimage.filters import threshold_otsu, threshold_local
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.morphology import disk
from skimage.segmentation import slic, mark_boundaries
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {len(tf.config.list_physical_devices("GPU")) > 0}')

TensorFlow : 2.20.0
GPU        : False


### Install Kaggle API client

In [4]:
pip install kaggle

### Authenticate Kaggle API
To download datasets from Kaggle, you need an API token. Follow these steps to get your `kaggle.json` file:

1.  Go to your Kaggle account page (kaggle.com/your_username/account).
2.  Scroll down to the 'API' section.
3.  Click on 'Create New API Token' to download `kaggle.json`.

Then, upload the `kaggle.json` file using the following code cell. Once uploaded, it will be moved to the correct directory.

In [ ]:
from google.colab import files

files.upload() # This will prompt you to upload the kaggle.json file

# Move kaggle.json to the appropriate directory
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

### Download and Unzip the Dataset
Now we can use the Kaggle API to download the 'New Plant Diseases Dataset' and then unzip it.

In [ ]:
# Download the dataset
# The dataset URL is: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset

# Unzip the dataset
import zipfile
import os

# Define the path to the downloaded zip file
zip_file_path = 'new-plant-diseases-dataset.zip'
# Define the directory where the dataset will be extracted
extract_path = 'new_plant_diseases_dataset_augmented'

# Create the extraction directory if it doesn't exist
if not os.path.exists(extract_path):
    os.makedirs(extract_path)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f'Dataset extracted to: {extract_path}')

# Verify the content
print('Contents of the extracted dataset:')
!ls {extract_path}

## 2. Dataset Paths & Configuration

In [ ]:
BASE_DIR   = 'new_plant_diseases_dataset_augmented/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
TRAIN_DIR  = os.path.join(BASE_DIR, 'train')
VALID_DIR  = os.path.join(BASE_DIR, 'valid')

IMG_SIZE   = 128
BATCH_SIZE = 32

classes    = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(classes)
print(f'Classes : {NUM_CLASSES}')
print(classes[:8], '...')


In [ ]:
BASE_DIR   = 'new_plant_diseases_dataset_augmented/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
TRAIN_DIR  = os.path.join(BASE_DIR, 'train')
VALID_DIR  = os.path.join(BASE_DIR, 'valid')

IMG_SIZE   = 128
BATCH_SIZE = 32

classes    = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(classes)
print(f'Classes : {NUM_CLASSES}')
print(classes[:8], '...')


## 3. Exploratory Data Analysis

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Count images per class
class_counts = {c: len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in classes}
df_counts = pd.DataFrame(list(class_counts.items()), columns=['Class','Count']).sort_values('Count', ascending=False)
print(f'Total training images : {df_counts["Count"].sum():,}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = ['#2ecc71' if 'healthy' in c else '#e74c3c' for c in df_counts['Class']]
axes[0].bar(range(len(df_counts)), df_counts['Count'], color=colors)
axes[0].set_xticks(range(len(df_counts)))
axes[0].set_xticklabels([c.split('___')[-1][:12] for c in df_counts['Class']], rotation=90, fontsize=6)
axes[0].set_title('Images per class (green=healthy, red=diseased)')

healthy = sum(1 for c in classes if 'healthy' in c)
diseased = NUM_CLASSES - healthy
axes[1].pie([healthy, diseased], labels=['Healthy classes','Diseased classes'],
            autopct='%1.1f%%', colors=['#2ecc71','#e74c3c'], startangle=90)
axes[1].set_title('Healthy vs Diseased class ratio')
plt.tight_layout(); plt.show()

## 4. CHAPTER 2 — Image Enhancement & Restoration
*Rehaussement et Restauration d'Images*

### 4.1 Histogram Analysis & Modification

In [ ]:
def get_sample_img(cls=None):
    cls = cls or random.choice(classes)
    d = os.path.join(TRAIN_DIR, cls)
    return cv2.cvtColor(cv2.imread(os.path.join(d, random.choice(os.listdir(d)))), cv2.COLOR_BGR2RGB), cls

img_rgb, cls_name = get_sample_img()
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

# --- Original ---
axes[0][0].imshow(img_rgb); axes[0][0].set_title('Original'); axes[0][0].axis('off')
axes[1][0].hist(img_gray.ravel(), bins=256, color='gray', alpha=0.7)
axes[1][0].set_title('Histogram original'); axes[1][0].set_xlabel('Intensity')

# --- Dynamic range extension (Contrast Stretching) Ch2 p19 ---
n0, n1 = img_gray.min(), img_gray.max()
contrast_stretched = ((img_gray.astype(np.float32) - n0) / (n1 - n0 + 1e-8) * 255).astype(np.uint8)
axes[0][1].imshow(contrast_stretched, cmap='gray'); axes[0][1].set_title('Contrast Stretching (Ch2)'); axes[0][1].axis('off')
axes[1][1].hist(contrast_stretched.ravel(), bins=256, color='blue', alpha=0.7)
axes[1][1].set_title('Histogram contrast stretched')

# --- Histogram Equalization (Ch2 p24-27) ---
img_eq = cv2.equalizeHist(img_gray)
axes[0][2].imshow(img_eq, cmap='gray'); axes[0][2].set_title('Histogram Equalization (Ch2)'); axes[0][2].axis('off')
axes[1][2].hist(img_eq.ravel(), bins=256, color='green', alpha=0.7)
axes[1][2].set_title('Histogram equalized')

# --- Image Inversion (Ch2 p23) ---
img_inv = 255 - img_gray
axes[0][3].imshow(img_inv, cmap='gray'); axes[0][3].set_title('Image Inversion (Ch2)'); axes[0][3].axis('off')
axes[1][3].hist(img_inv.ravel(), bins=256, color='red', alpha=0.7)
axes[1][3].set_title('Histogram inverted')

plt.suptitle(f'Ch2 – Histogram Techniques | {cls_name}', fontsize=12)
plt.tight_layout(); plt.show()

### 4.2 Manual Histogram Equalization (4-step algorithm, Ch2 p27)

In [ ]:
def manual_histogram_equalization(img_gray):
    """Ch2 - 4-step histogram equalization algorithm"""
    L = 256
    # Step 1: Compute histogram
    hist, bins = np.histogram(img_gray.ravel(), bins=L, range=(0, L))
    # Step 2: Compute cumulative histogram
    cum_hist = np.cumsum(hist)
    # Step 3: Normalize and multiply by max gray level
    total_pixels = img_gray.size
    cum_normalized = (cum_hist / total_pixels) * (L - 1)
    # Step 4: Map normalized values back to gray levels
    equalized = cum_normalized[img_gray].astype(np.uint8)
    return equalized, hist, cum_normalized

eq_img, hist_orig, cum_norm = manual_histogram_equalization(img_gray)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes[0][0].imshow(img_gray, cmap='gray'); axes[0][0].set_title('Original'); axes[0][0].axis('off')
axes[0][1].bar(range(256), hist_orig, color='steelblue', width=1); axes[0][1].set_title('Step 1: Original Histogram')
axes[0][2].plot(cum_norm, color='orange'); axes[0][2].set_title('Step 2-3: Cumulative Normalized')
axes[1][0].imshow(eq_img, cmap='gray'); axes[1][0].set_title('Step 4: Equalized Image'); axes[1][0].axis('off')
axes[1][1].hist(eq_img.ravel(), bins=256, color='green', width=1); axes[1][1].set_title('Equalized Histogram')
axes[1][2].plot(np.cumsum(np.histogram(eq_img.ravel(), 256)[0]), color='purple')
axes[1][2].set_title('Equalized CDF (flatter)')
plt.suptitle('Ch2 – Manual 4-Step Histogram Equalization Algorithm', fontsize=12)
plt.tight_layout(); plt.show()
print('Verification: histogram is flatter after equalization ✓')

### 4.3 Spatial Filtering — Linear (Mean, Gaussian) & Non-linear (Median) — Ch2 p39-46

In [ ]:
# Add synthetic noise to demonstrate filtering
noise = np.random.normal(0, 25, img_gray.shape).astype(np.float32)
img_noisy = np.clip(img_gray.astype(np.float32) + noise, 0, 255).astype(np.uint8)

# --- Ch2 p42: Mean (averaging) filter (low-pass) ---
kernel_mean_3 = np.ones((3,3), np.float32) / 9
kernel_mean_5 = np.ones((5,5), np.float32) / 25
img_mean3 = cv2.filter2D(img_noisy, -1, kernel_mean_3)       # manual 2D convolution
img_mean5 = cv2.filter2D(img_noisy, -1, kernel_mean_5)

# --- Ch2 p42: Gaussian filter ---
img_gauss3  = cv2.GaussianBlur(img_noisy, (3,3),  sigmaX=1)
img_gauss7  = cv2.GaussianBlur(img_noisy, (7,7),  sigmaX=3)

# --- Ch2 p45: Non-linear Median filter ---
img_median3 = cv2.medianBlur(img_noisy, 3)
img_median7 = cv2.medianBlur(img_noisy, 7)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
imgs = [img_noisy, img_mean3, img_mean5, img_gauss7,
        img_median3, img_median7, img_mean3, img_median3]
titles = ['Noisy input','Mean 3x3 (Ch2)','Mean 5x5 (Ch2)','Gaussian σ=3 (Ch2)',
          'Median 3x3 (Ch2)','Median 7x7 (Ch2)','Mean vs Median','Comparison']
cmaps  = ['gray']*8
for ax, im, t in zip(axes.flatten(), imgs, titles):
    ax.imshow(im, cmap='gray'); ax.set_title(t, fontsize=9); ax.axis('off')

# Replace last two with diff maps
axes[1][2].imshow(np.abs(img_noisy.astype(int)-img_mean3.astype(int)), cmap='hot')
axes[1][2].set_title('Noise removed by Mean', fontsize=9); axes[1][2].axis('off')
axes[1][3].imshow(np.abs(img_noisy.astype(int)-img_median3.astype(int)), cmap='hot')
axes[1][3].set_title('Noise removed by Median', fontsize=9); axes[1][3].axis('off')

plt.suptitle('Ch2 – Linear (Mean/Gaussian) and Non-Linear (Median) Filtering', fontsize=12)
plt.tight_layout(); plt.show()

## 5. CHAPTER 3 — Mathematical Morphology
*Morphologie Mathématique*

### 5.1 Erosion, Dilation, Opening, Closing

In [ ]:
# Work on binary version of leaf
_, img_bin = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Structuring element (Ch3)
se3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
se7 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7))

# Ch3 p19-21: Erosion
erosion3  = cv2.erode(img_bin, se3)
erosion7  = cv2.erode(img_bin, se7)

# Ch3 p22-24: Dilation
dilation3 = cv2.dilate(img_bin, se3)
dilation7 = cv2.dilate(img_bin, se7)

# Ch3 p27: Opening = erosion then dilation
opening   = cv2.morphologyEx(img_bin, cv2.MORPH_OPEN,  se7)

# Ch3 p29: Closing = dilation then erosion
closing   = cv2.morphologyEx(img_bin, cv2.MORPH_CLOSE, se7)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
data = [img_bin, erosion3, erosion7, dilation3,
        dilation7, opening, closing, cv2.bitwise_xor(opening, closing)]
labels = ['Binary (Otsu)','Erosion 3x3','Erosion 7x7','Dilation 3x3',
          'Dilation 7x7','Opening 7x7','Closing 7x7','Opening XOR Closing']
for ax, im, t in zip(axes.flatten(), data, labels):
    ax.imshow(im, cmap='gray'); ax.set_title(t, fontsize=9); ax.axis('off')
plt.suptitle('Ch3 – Morphological Operations (Erosion, Dilation, Opening, Closing)', fontsize=12)
plt.tight_layout(); plt.show()

### 5.2 Morphological Gradients + Tophat Transform

In [ ]:
# Working on grayscale
se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
eroded  = cv2.erode(img_gray,  se)
dilated = cv2.dilate(img_gray, se)

# Ch3 p26: Gradients morphologiques
grad_internal  = cv2.subtract(img_gray, eroded)             # X - (X ⊖ B)
grad_external  = cv2.subtract(dilated, img_gray)            # (X ⊕ B) - X
grad_morpho    = cv2.subtract(dilated, eroded)              # (X ⊕ B) - (X ⊖ B)

# Ch3 p34: White Tophat = X - Opening(X)  → bright small features
opening_gray   = cv2.morphologyEx(img_gray, cv2.MORPH_OPEN,  se)
closing_gray   = cv2.morphologyEx(img_gray, cv2.MORPH_CLOSE, se)
white_tophat   = cv2.subtract(img_gray, opening_gray)       # Tw(X) = X - X∘B
black_tophat   = cv2.subtract(closing_gray, img_gray)       # Tb(X) = X•B - X

# Tophat enhanced image (highlights disease spots)
enhanced       = cv2.add(img_gray, white_tophat)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
data   = [img_gray, grad_internal, grad_external, grad_morpho,
          white_tophat, black_tophat, enhanced, opening_gray]
labels = ['Original gray','Internal Gradient','External Gradient','Morphological Gradient',
          'White Tophat (disease spots)','Black Tophat','Tophat Enhanced','Opening']
cmaps  = ['gray']*8
for ax, im, t in zip(axes.flatten(), data, labels):
    ax.imshow(im, cmap='gray'); ax.set_title(t, fontsize=9); ax.axis('off')
plt.suptitle('Ch3 – Morphological Gradients & Tophat Transforms', fontsize=12)
plt.tight_layout(); plt.show()
print('White Tophat highlights bright disease spots on leaves ✓')

## 6. CHAPTER 4 — Image Segmentation
*Segmentation d'Images*

### 6.1 Edge-based Segmentation — Sobel, Laplacian, Canny

In [ ]:
# Smooth first to reduce noise before edge detection
img_smooth = cv2.GaussianBlur(img_gray, (5,5), sigmaX=1)

# Ch4 p11-13: Gradient approach — Sobel
sobel_x   = cv2.Sobel(img_smooth, cv2.CV_64F, 1, 0, ksize=3)
sobel_y   = cv2.Sobel(img_smooth, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)
sobel_mag = (sobel_mag / sobel_mag.max() * 255).astype(np.uint8)

# Prewitt (also mentioned in Ch4 exercise)
kernel_prewitt_x = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32)
kernel_prewitt_y = np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=np.float32)
prewitt_x = cv2.filter2D(img_smooth.astype(np.float32), -1, kernel_prewitt_x)
prewitt_y = cv2.filter2D(img_smooth.astype(np.float32), -1, kernel_prewitt_y)
prewitt   = np.clip(np.sqrt(prewitt_x**2 + prewitt_y**2), 0, 255).astype(np.uint8)

# Ch4 p18: Laplacian (second derivative)
laplacian_kernel = np.array([[0,1,0],[1,-4,1],[0,1,0]], dtype=np.float32)  # 3x3 approx from Ch4
laplacian = cv2.Laplacian(img_smooth, cv2.CV_64F)
laplacian = np.uint8(np.absolute(laplacian))

# Ch4 p16: Canny (multi-step edge detector)
canny     = cv2.Canny(img_smooth, 50, 150)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
data   = [img_rgb, sobel_mag, prewitt, laplacian, canny,
          np.stack([sobel_mag, np.zeros_like(sobel_mag), canny], axis=-1)]
labels = ['Original','Sobel (Ch4)','Prewitt (Ch4 exercise)','Laplacian 2nd deriv (Ch4)','Canny (Ch4)',
          'Sobel(R) + Canny(B) overlay']
for ax, im, t in zip(axes.flatten(), data, labels):
    ax.imshow(im if im.ndim==3 else im, cmap=None if im.ndim==3 else 'gray')
    ax.set_title(t, fontsize=9); ax.axis('off')
plt.suptitle('Ch4 – Edge-based Segmentation (Sobel, Prewitt, Laplacian, Canny)', fontsize=12)
plt.tight_layout(); plt.show()

### 6.2 Segmentation by Thresholding — Otsu Method (Ch4 p29-30)

In [ ]:
def otsu_manual(img_gray):
    """Manual Otsu algorithm (Ch4 p29-30): maximize inter-class variance"""
    L = 256
    hist, _ = np.histogram(img_gray.ravel(), bins=L, range=(0, L))
    total   = img_gray.size
    prob    = hist / total

    best_t, best_var = 0, 0
    for t in range(1, L):
        w0 = prob[:t].sum()   # weight class 0 (background)
        w1 = prob[t:].sum()   # weight class 1 (foreground)
        if w0 == 0 or w1 == 0: continue
        mu0 = (np.arange(t)   * prob[:t]).sum() / w0
        mu1 = (np.arange(t,L) * prob[t:]).sum() / w1
        var_between = w0 * w1 * (mu0 - mu1)**2  # σ²_B: inter-class variance
        if var_between > best_var:
            best_var, best_t = var_between, t
    return best_t

T_manual = otsu_manual(img_gray)
T_skimage = threshold_otsu(img_gray)

print(f'Otsu threshold (manual)  : {T_manual}')
print(f'Otsu threshold (skimage) : {T_skimage}')

img_otsu_global    = (img_gray > T_manual).astype(np.uint8) * 255
# Local adaptive thresholding
T_local            = threshold_local(img_gray, block_size=35, offset=10)
img_otsu_local     = (img_gray > T_local).astype(np.uint8) * 255
# Adaptive (OpenCV)
img_adaptive       = cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                           cv2.THRESH_BINARY, 35, 10)

# Noise sensitivity demo (Ch4 p28)
img_noisy2 = np.clip(img_gray.astype(int) + np.random.normal(0,40,img_gray.shape), 0, 255).astype(np.uint8)
T_noisy    = otsu_manual(img_noisy2)
img_noisy_seg = (img_noisy2 > T_noisy).astype(np.uint8) * 255

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
data   = [img_gray, img_otsu_global, img_otsu_local,
          img_adaptive, img_noisy2, img_noisy_seg]
labels = [f'Original gray',
          f'Global Otsu T={T_manual} (Ch4)',
          'Local Thresholding (Ch4)',
          'Adaptive Threshold (Ch4)',
          'Noisy image (+σ40)',
          f'Otsu on noisy (Ch4 – sensitivity demo)']
for ax, im, t in zip(axes.flatten(), data, labels):
    ax.imshow(im, cmap='gray'); ax.set_title(t, fontsize=9); ax.axis('off')
plt.suptitle('Ch4 – Thresholding & Otsu Method (manual implementation)', fontsize=12)
plt.tight_layout(); plt.show()

### 6.3 K-Means Color Segmentation (Ch4 p46-50)

In [ ]:
def kmeans_segment(img_rgb, k=3):
    """Ch4 p46-50: K-means segmentation (unsupervised classification)"""
    pixels = img_rgb.reshape(-1, 3).astype(np.float32)
    # OpenCV K-means
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    _, labels, centers = cv2.kmeans(pixels, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    centers = np.uint8(centers)
    segmented = centers[labels.flatten()].reshape(img_rgb.shape)
    return segmented, labels.reshape(img_rgb.shape[:2])

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes[0][0].imshow(img_rgb); axes[0][0].set_title('Original'); axes[0][0].axis('off')
for idx, k in enumerate([2, 3, 4, 5, 6]):
    seg, lbl = kmeans_segment(img_rgb, k)
    ax = axes.flatten()[idx+1]
    ax.imshow(seg); ax.set_title(f'K-Means k={k} (Ch4)'); ax.axis('off')

plt.suptitle('Ch4 – K-Means Color Segmentation (unsupervised classification)', fontsize=12)
plt.tight_layout(); plt.show()

# Show diseased region isolation with k=3
seg3, lbl3 = kmeans_segment(img_rgb, 3)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img_rgb); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(seg3); axes[1].set_title('K=3 segmented'); axes[1].axis('off')
for i, ax in enumerate(axes[2:]):
    mask = (lbl3 == i).astype(np.uint8) * 255
    ax.imshow(mask, cmap='gray'); ax.set_title(f'Region {i+1}'); ax.axis('off')
plt.suptitle('Ch4 – K-Means regions (each cluster = one segment)', fontsize=12)
plt.tight_layout(); plt.show()

## 7. CHAPTER 5 — Texture Analysis
*Analyse de Texture*

### 7.1 GLCM — Gray-Level Co-occurrence Matrix + Haralick Features (Ch5 p9-11)

In [ ]:
def extract_glcm_features(img_gray, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4]):
    """Ch5 p9-11: GLCM + Haralick features
    Computes: contrast, dissimilarity, homogeneity, energy, correlation, ASM"""
    # Quantize to 64 levels for efficiency
    img_q = (img_gray / 4).astype(np.uint8)
    glcm  = graycomatrix(img_q, distances=distances, angles=angles,
                         levels=64, symmetric=True, normed=True)
    features = {}
    for prop in ['contrast','dissimilarity','homogeneity','energy','correlation','ASM']:
        val = graycoprops(glcm, prop)
        features[f'glcm_{prop}_mean'] = val.mean()
        features[f'glcm_{prop}_range'] = val.max() - val.min()
    return features, glcm

feats, glcm = extract_glcm_features(img_gray)
print('GLCM / Haralick Features (Ch5):')
for k, v in feats.items():
    print(f'  {k:<35} = {v:.5f}')

# Visualize GLCM matrices for 4 directions
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(img_gray, cmap='gray'); axes[0].set_title('Leaf image'); axes[0].axis('off')
directions = ['0°','45°','90°','135°']
for i, (ax, d) in enumerate(zip(axes[1:], directions)):
    matrix = glcm[:,:,0,i]
    im = ax.imshow(np.log(matrix + 1e-6), cmap='viridis')
    ax.set_title(f'GLCM {d} (Ch5)'); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Ch5 – GLCM Co-occurrence Matrices (4 directions)', fontsize=12)
plt.tight_layout(); plt.show()

### 7.2 LBP — Local Binary Pattern (Ch5 p14-16)

In [ ]:
def extract_lbp(img_gray, P=8, R=1):
    """Ch5 p14-16: LBP operator"""
    lbp_img = local_binary_pattern(img_gray, P, R, method='uniform')
    n_bins  = int(lbp_img.max()) + 1
    hist, _ = np.histogram(lbp_img.ravel(), bins=n_bins, range=(0, n_bins), density=True)
    return lbp_img, hist

# Compare healthy vs diseased
healthy_cls  = [c for c in classes if 'healthy' in c][0]
diseased_cls = [c for c in classes if 'healthy' not in c][0]

def load_gray(cls):
    d = os.path.join(TRAIN_DIR, cls)
    return cv2.cvtColor(cv2.imread(os.path.join(d, os.listdir(d)[0])), cv2.COLOR_BGR2GRAY)

h_img  = load_gray(healthy_cls)
d_img  = load_gray(diseased_cls)
h_lbp, h_hist = extract_lbp(h_img)
d_lbp, d_hist = extract_lbp(d_img)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes[0][0].imshow(h_img, cmap='gray'); axes[0][0].set_title(f'Healthy: {healthy_cls[:30]}'); axes[0][0].axis('off')
axes[0][1].imshow(h_lbp, cmap='jet');  axes[0][1].set_title('LBP Map (healthy)');  axes[0][1].axis('off')
axes[0][2].bar(range(len(h_hist)), h_hist, color='green', alpha=0.7)
axes[0][2].set_title('LBP Histogram (healthy)')

axes[1][0].imshow(d_img, cmap='gray'); axes[1][0].set_title(f'Diseased: {diseased_cls[:30]}'); axes[1][0].axis('off')
axes[1][1].imshow(d_lbp, cmap='jet');  axes[1][1].set_title('LBP Map (diseased)'); axes[1][1].axis('off')
axes[1][2].bar(range(len(d_hist)), d_hist, color='red', alpha=0.7)
axes[1][2].set_title('LBP Histogram (diseased)')

plt.suptitle('Ch5 – LBP: Local Binary Pattern comparison (Healthy vs Diseased)', fontsize=12)
plt.tight_layout(); plt.show()
print('LBP histograms differ between healthy and diseased → texture is discriminative ✓')

### 7.3 Wavelet Analysis (Ch5 p17)

In [ ]:
# Ch5 p17: Spatio-frequency approach — Wavelets
img_resized = cv2.resize(img_gray, (128, 128))

# 2D Discrete Wavelet Transform — Haar wavelet
coeffs2 = pywt.dwt2(img_resized, 'haar')
cA, (cH, cV, cD) = coeffs2

# 2-level decomposition
coeffs2_l2 = pywt.dwt2(cA, 'haar')
cA2, (cH2, cV2, cD2) = coeffs2_l2

def norm(c):
    c = np.abs(c)
    return ((c - c.min()) / (c.max() - c.min() + 1e-8) * 255).astype(np.uint8)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes[0][0].imshow(img_resized, cmap='gray'); axes[0][0].set_title('Original (128x128)'); axes[0][0].axis('off')
for ax, coef, name in zip(axes[0][1:], [cA, cH, cV, cD],
                           ['Approx (LL)','Horiz detail (LH)','Vert detail (HL)','Diag detail (HH)']):
    ax.imshow(norm(coef), cmap='gray'); ax.set_title(f'Level 1: {name}'); ax.axis('off')

axes[1][0].imshow(img_resized, cmap='gray'); axes[1][0].set_title('Level 2 decomp'); axes[1][0].axis('off')
for ax, coef, name in zip(axes[1][1:], [cA2, cH2, cV2, cD2],
                           ['Approx L2','H-detail L2','V-detail L2','D-detail L2']):
    ax.imshow(norm(coef), cmap='gray'); ax.set_title(name, fontsize=9); ax.axis('off')

plt.suptitle('Ch5 – Wavelet Analysis (Haar DWT, 2-level decomposition)', fontsize=12)
plt.tight_layout(); plt.show()

# Wavelet energy features
wavelet_features = {
    'wavelet_energy_LL': np.sum(cA**2),
    'wavelet_energy_LH': np.sum(cH**2),
    'wavelet_energy_HL': np.sum(cV**2),
    'wavelet_energy_HH': np.sum(cD**2),
}
print('Wavelet energy features:')
for k,v in wavelet_features.items(): print(f'  {k} = {v:.2f}')

## 8. Preprocessing Pipeline for CNN
*Applies Ch2-5 techniques as preprocessing steps before training*

In [ ]:
# Training augmentation (with CLAHE histogram equalization — Ch2)
def preprocess_image(img_array_uint8):
    """Apply Ch2 preprocessing: CLAHE equalization + Gaussian denoising"""
    lab = cv2.cvtColor(img_array_uint8, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    lab[:,:,0] = clahe.apply(lab[:,:,0])   # equalize L channel only
    result = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    result = cv2.GaussianBlur(result, (3,3), 0)  # Ch2 Gaussian filter
    return result

# Standard generators with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True, seed=42)

valid_gen = valid_datagen.flow_from_directory(
    VALID_DIR, target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

class_names = list(train_gen.class_indices.keys())
print(f'Train batches : {len(train_gen)}')
print(f'Valid batches : {len(valid_gen)}')

## 9. Model 1 — Custom CNN from Scratch

In [ ]:
def build_cnn(num_classes, input_shape=(128,128,3)):
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32,(3,3),activation='relu',padding='same',input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32,(3,3),activation='relu',padding='same'),
        layers.MaxPooling2D(2,2), layers.Dropout(0.25),
        # Block 2
        layers.Conv2D(64,(3,3),activation='relu',padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64,(3,3),activation='relu',padding='same'),
        layers.MaxPooling2D(2,2), layers.Dropout(0.25),
        # Block 3
        layers.Conv2D(128,(3,3),activation='relu',padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128,(3,3),activation='relu',padding='same'),
        layers.MaxPooling2D(2,2), layers.Dropout(0.3),
        # Block 4
        layers.Conv2D(256,(3,3),activation='relu',padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2), layers.Dropout(0.3),
        # Head
        layers.GlobalAveragePooling2D(),
        layers.Dense(512,activation='relu'),
        layers.BatchNormalization(), layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name='CustomCNN')
    return model

cnn_model = build_cnn(NUM_CLASSES)
cnn_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()

In [ ]:
cb_cnn = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_cnn.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]
history_cnn = cnn_model.fit(train_gen, epochs=15, validation_data=valid_gen, callbacks=cb_cnn)

## 10. Model 2 — Transfer Learning with EfficientNetB3

In [ ]:
def build_efficientnet(num_classes, input_shape=(128,128,3)):
    base = EfficientNetB3(include_top=False, weights='imagenet', input_shape=input_shape)
    base.trainable = False
    inp = keras.Input(shape=input_shape)
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(512, activation='relu')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inp, out, name='EfficientNetB3'), base

tl_model, base_model = build_efficientnet(NUM_CLASSES)
tl_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss='categorical_crossentropy', metrics=['accuracy'])
tl_model.summary()

In [ ]:
cb_tl = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_tl.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]
# Phase 1 — frozen base
history_tl1 = tl_model.fit(train_gen, epochs=10, validation_data=valid_gen, callbacks=cb_tl)

In [ ]:
# Phase 2 — fine-tune last 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]: layer.trainable = False
tl_model.compile(optimizer=keras.optimizers.Adam(1e-4),
                 loss='categorical_crossentropy', metrics=['accuracy'])
history_tl2 = tl_model.fit(train_gen, epochs=20, validation_data=valid_gen, callbacks=cb_tl)

## 11. Training Curves

In [ ]:
def plot_history(history, title):
    acc, val_acc = history.history['accuracy'], history.history['val_accuracy']
    loss, val_loss = history.history['loss'], history.history['val_loss']
    ep = range(1, len(acc)+1)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    a1.plot(ep, acc, 'b-o', markersize=4, label='Train'); a1.plot(ep, val_acc, 'r-o', markersize=4, label='Val')
    a1.set_title(f'{title} – Accuracy'); a1.legend(); a1.grid(alpha=0.3)
    a2.plot(ep, loss, 'b-o', markersize=4, label='Train'); a2.plot(ep, val_loss, 'r-o', markersize=4, label='Val')
    a2.set_title(f'{title} – Loss'); a2.legend(); a2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

plot_history(history_cnn, 'Custom CNN')
plot_history(history_tl2, 'EfficientNetB3')

## 12. Evaluation — Confusion Matrix & Classification Report

In [ ]:
print('=== Custom CNN ===')
l1, a1 = cnn_model.evaluate(valid_gen, verbose=0)
print(f'  Val Accuracy: {a1*100:.2f}%  Loss: {l1:.4f}')

print('\n=== EfficientNetB3 ===')
l2, a2 = tl_model.evaluate(valid_gen, verbose=0)
print(f'  Val Accuracy: {a2*100:.2f}%  Loss: {l2:.4f}')

valid_gen.reset()
y_pred = np.argmax(tl_model.predict(valid_gen, verbose=1), axis=1)
y_true = valid_gen.classes

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(22,18))
sns.heatmap(cm, annot=False, cmap='YlOrRd',
            xticklabels=[c.split('___')[-1][:12] for c in class_names],
            yticklabels=[c.split('___')[-1][:12] for c in class_names])
plt.title('Confusion Matrix — EfficientNetB3', fontsize=14)
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=90, fontsize=7); plt.yticks(fontsize=7)
plt.tight_layout(); plt.show()

short = [c.replace('___',' | ').replace('_',' ') for c in class_names]
print(classification_report(y_true, y_pred, target_names=short))

## 13. Grad-CAM Visualization
*What features does the model look at — links back to Ch3 tophat / Ch4 edge detection*

In [ ]:
def gradcam_heatmap(model, img_array, layer_name):
    grad_model = tf.keras.models.Model([model.inputs],
        [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        pred_idx = tf.argmax(preds[0])
        loss = preds[:, pred_idx]
    grads = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    conv_out = conv_out[0]
    heatmap = conv_out @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap)+1e-8)
    return heatmap.numpy(), int(pred_idx)

try:
    last_conv = 'top_activation'
    tl_model.get_layer(last_conv)
except:
    last_conv = [l.name for l in tl_model.layers if 'conv' in l.name.lower()][-1]

print(f'Grad-CAM layer: {last_conv}')

samples = []
for cls in random.sample(class_names, 4):
    d = os.path.join(VALID_DIR, cls)
    f = random.choice(os.listdir(d))
    samples.append((os.path.join(d,f), cls))

fig, axes = plt.subplots(len(samples), 3, figsize=(12, 4*len(samples)))
for i, (path, true_cls) in enumerate(samples):
    orig = np.array(load_img(path, target_size=(IMG_SIZE,IMG_SIZE)))
    arr  = np.expand_dims(orig/255.0, 0).astype('float32')
    try:
        hm, pred_idx = gradcam_heatmap(tl_model, arr, last_conv)
        hm_r = cv2.resize(hm, (IMG_SIZE, IMG_SIZE))
        hm_c = cv2.applyColorMap(np.uint8(255*hm_r), cv2.COLORMAP_JET)
        hm_c = cv2.cvtColor(hm_c, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(orig, 0.6, hm_c, 0.4, 0)
        pred_name = class_names[pred_idx].split('___')[-1][:20]
        true_name = true_cls.split('___')[-1][:20]
        color = 'green' if pred_idx == list(class_names).index(true_cls) else 'red'
        axes[i][0].imshow(orig); axes[i][0].set_title(f'True: {true_name}', fontsize=8); axes[i][0].axis('off')
        axes[i][1].imshow(hm_r, cmap='jet'); axes[i][1].set_title('Heatmap', fontsize=8); axes[i][1].axis('off')
        axes[i][2].imshow(overlay); axes[i][2].set_title(f'Pred: {pred_name}', fontsize=8, color=color); axes[i][2].axis('off')
    except Exception as e:
        print(f'Skipped {true_cls}: {e}')

plt.suptitle('Grad-CAM — model focuses on disease spots (like Ch3 Tophat + Ch4 edges)', fontsize=12)
plt.tight_layout(); plt.show()

## 14. Full Pipeline Summary

In [ ]:
summary = {
    'Chapter': ['Ch2','Ch2','Ch2','Ch3','Ch3','Ch3','Ch4','Ch4','Ch4','Ch5','Ch5','Ch5','CNN'],
    'Technique': [
        'Histogram Equalization (CLAHE)','Mean/Gaussian Filter','Median Filter',
        'Erosion & Dilation','Opening & Closing','Tophat Transform',
        'Sobel/Canny/Laplacian edges','Otsu Thresholding','K-Means Segmentation',
        'GLCM / Haralick features','LBP texture','Wavelet decomposition',
        'EfficientNetB3 Transfer Learning'
    ],
    'Role in project': [
        'Preprocess images before CNN','Denoise leaf images','Denoise (non-linear)',
        'Morphological analysis','Disease spot isolation','Highlight disease spots',
        'Edge analysis','Isolate diseased regions','Color-based segmentation',
        'Texture discrimination','Texture pattern recognition','Frequency analysis',
        'Disease classification'
    ]
}
df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

print(f'\n=== Final Results ===')
print(f'Custom CNN Accuracy    : {a1*100:.2f}%')
print(f'EfficientNetB3 Accuracy: {a2*100:.2f}%')

## 15. Save Best Model

In [ ]:
tl_model.save('crop_disease_efficientnet_final.keras')
with open('class_names.json','w') as f: json.dump(class_names, f)
print(f'Model saved. Classes: {len(class_names)}')